# Serif Takehome assignment

What is the list of machine readable file URLs that represent the Anthem PPO network in New York state?

Your output should be the list of machine readable file URLs corresponding to Anthem's PPO in New York state.

Here's the development workflow: 1. Write a rough draft with my thoughts as I go, 2. Make a cleaned up 'main' notebook from it, and 3. Produce a `main.py` script from the 'main' notebook.

This is the "draft" notebook.

The rough plan, as I see it:

1. Open the `json` and read it in **chunks** (or as stream). It's only 24G  uncompressed, but let's pretend this will have to extend to a 1TB file. 

2. [Refer to the published schema](https://github.com/CMSgov/price-transparency-guide/tree/master/schemas/table-of-contents) but anticipate getting something different. (This is why we do this in Jupyter!)

3. Handle multiply repeated data using Python's `set`. If performance is a concern, an XOR filter (or bloom filter; whatever a good library provides) is usually faster for checking set membership, at the risk (which can be made arbitrarily tiny) of false-positively rejecting a novel datapoint as new.

# Getting started

Let's get our imports out of the way. 

Read from`2026-02-01_anthem_index.json.gz`, chunking to avoid running out of memory. (It's ~11GB compressed and RAM is expensive)

In [7]:
# first, extract the json file 
!gzip -c -d 2026-02-01_anthem_index.json.gz > index.json

In [8]:
# Add dependencies as needed to the `pyproject.toml`.
!uv add pandas

Resolved 6 packages in 7ms
Audited 4 packages in 1ms


In [5]:
import json
import os, sys
import pandas as pd

Resolved 6 packages in 4ms
Audited 4 packages in 0.14ms


# Chunking the JSON

I've never actually had to chunk a large JSON before. This is more complicated than 